# Chunking Strategies for RAG Systems

**FAMNIT AI Workshop -- Day 3**

In Day 2, we learned how to embed text and store it in a vector database. But before we can embed a document, we need to **split it into smaller pieces** called *chunks*. The way we chunk text has a huge impact on the quality of our search results.

In this notebook, we will explore and compare different chunking strategies hands-on.

---
## Section 1: Why Chunking Matters

### The Problem

Embedding models have **token limits** (e.g., 256 tokens for MiniLM, 512 for many others). We cannot feed an entire document into them at once.

Even if we could, there is a fundamental trade-off:

| Chunk size | Precision | Context |
|---|---|---|
| **Small** chunks | High -- each chunk is about one specific thing | Low -- may lose surrounding context |
| **Large** chunks | Low -- the embedding averages over many topics | High -- more context is preserved |

The goal is to find the **sweet spot**: chunks that are small enough to be precise, but large enough to be meaningful.

### The Pipeline

```
 +------------+      +-----------+      +------------+      +-----------+
 |            |      |           |      |            |      |           |
 |  Document  +----->+  Chunks   +----->+ Embeddings +----->+ Vector DB |
 |            |      |           |      |            |      |           |
 +------------+      +-----------+      +------------+      +-----------+
                          ^
                          |
                   THIS is what we
                   focus on today!
```

Chunking is the **first and most important** preprocessing step. Bad chunking = bad retrieval = bad answers.

In [ ]:
# First, let's install the packages we need (run once)
# !pip install langchain-text-splitters sentence-transformers matplotlib numpy pandas scikit-learn

In [ ]:
# Our sample text about Artificial Intelligence -- we will use this throughout the notebook

SAMPLE_TEXT = """
Artificial intelligence (AI) is a branch of computer science that aims to create \
intelligent machines capable of performing tasks that typically require human intelligence. \
These tasks include learning from experience, understanding natural language, recognizing \
patterns, and making decisions. The field was founded in the 1950s by pioneers such as \
Alan Turing, John McCarthy, and Marvin Minsky, who believed that machines could be made \
to simulate any aspect of human intelligence.

Machine learning, a subset of AI, focuses on developing algorithms that allow computers \
to learn from and make predictions based on data. Unlike traditional programming, where \
rules are explicitly coded, machine learning systems improve their performance through \
experience. Deep learning, a further subset, uses artificial neural networks with many \
layers to model complex patterns in large datasets. These techniques have led to \
breakthroughs in image recognition, speech processing, and natural language understanding.

Today, AI is transforming virtually every industry. In healthcare, AI systems assist \
doctors in diagnosing diseases from medical images with remarkable accuracy. In finance, \
algorithms detect fraudulent transactions in real time. Self-driving cars use AI to \
navigate roads safely. Large language models like GPT and Claude can generate human-like \
text, translate languages, summarize documents, and answer complex questions. As AI \
continues to advance, ethical considerations around bias, privacy, and job displacement \
have become increasingly important topics of discussion in both academia and industry.
""".strip()

print(f"Sample text length: {len(SAMPLE_TEXT)} characters")
print(f"Sample text preview: {SAMPLE_TEXT[:100]}...")

---
## Section 2: Fixed-Size Chunking

The simplest approach: split text into chunks of a fixed number of characters, optionally with **overlap** between consecutive chunks.

```
Text: |AAAAAAAAAA|BBBBBBBBBB|CCCCCCCCCC|

Without overlap (chunk_size=10, overlap=0):
  Chunk 1: |AAAAAAAAAA|
  Chunk 2:            |BBBBBBBBBB|
  Chunk 3:                       |CCCCCCCCCC|

With overlap (chunk_size=10, overlap=3):
  Chunk 1: |AAAAAAAAAA|
  Chunk 2:        |AAABBBBBBB|
  Chunk 3:               |BBBCCCCCCC|
                  ^^^          ^^^
               overlapping   overlapping
               region        region
```

**Overlap** helps ensure that information at the boundary between two chunks is not lost.

In [ ]:
def fixed_size_chunk(text, chunk_size=200, overlap=50):
    """
    Split text into fixed-size chunks with optional overlap.
    
    Args:
        text: The input text to chunk.
        chunk_size: The maximum number of characters per chunk.
        overlap: The number of characters to overlap between consecutive chunks.
    
    Returns:
        A list of text chunks.
    """
    if chunk_size <= overlap:
        raise ValueError("chunk_size must be greater than overlap")
    
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap  # step forward by (chunk_size - overlap)
    
    return chunks

print("Function defined!")

In [ ]:
# Apply fixed-size chunking to our sample text

fixed_chunks = fixed_size_chunk(SAMPLE_TEXT, chunk_size=200, overlap=50)

print(f"Number of chunks: {len(fixed_chunks)}\n")
for i, chunk in enumerate(fixed_chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()

In [ ]:
import matplotlib.pyplot as plt

# Visualize chunk sizes
sizes = [len(c) for c in fixed_chunks]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(range(1, len(sizes) + 1), sizes, color='steelblue', edgecolor='black')
ax.set_xlabel('Chunk Number')
ax.set_ylabel('Chunk Size (characters)')
ax.set_title('Fixed-Size Chunking: Chunk Sizes (chunk_size=200, overlap=50)')
ax.set_xticks(range(1, len(sizes) + 1))
ax.axhline(y=200, color='red', linestyle='--', label='Target chunk size (200)')
ax.legend()

# Add size labels on top of each bar
for bar, size in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
            str(size), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

### Exercise 1

Try changing the parameters:

1. Set `chunk_size=100` and `overlap=20`. What happens to the number of chunks? Are they still meaningful?
2. Set `chunk_size=500` and `overlap=100`. How does this compare?
3. What happens if you set `overlap=0`? Can you think of a case where this would be a problem?

Modify the code cell above and re-run it to see the results.

---
## Section 3: Sentence-Based Chunking

Instead of splitting at arbitrary character positions, we can split at **sentence boundaries**. This preserves the grammatical structure of the text.

The idea: use punctuation marks (`.`, `!`, `?`) as natural split points.

In [ ]:
import re

def sentence_chunk(text):
    """
    Split text into chunks based on sentence boundaries.
    Uses regex to detect sentence-ending punctuation followed by whitespace.
    
    Returns:
        A list of sentences (each sentence is one chunk).
    """
    # Split after . or ! or ? that are followed by whitespace
    sentences = re.split(r'(?<=[.!?])\s+', text)
    # Remove any empty strings
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences

sentence_chunks = sentence_chunk(SAMPLE_TEXT)

print(f"Number of sentence chunks: {len(sentence_chunks)}\n")
for i, chunk in enumerate(sentence_chunks):
    print(f"--- Sentence {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()

In [ ]:
# Compare chunk count and sizes: Fixed-Size vs Sentence-Based

fixed_sizes = [len(c) for c in fixed_chunks]
sentence_sizes = [len(c) for c in sentence_chunks]

print("=" * 50)
print(f"{'Metric':<25} {'Fixed-Size':>12} {'Sentence':>12}")
print("=" * 50)
print(f"{'Number of chunks':<25} {len(fixed_chunks):>12} {len(sentence_chunks):>12}")
print(f"{'Average size (chars)':<25} {sum(fixed_sizes)/len(fixed_sizes):>12.1f} {sum(sentence_sizes)/len(sentence_sizes):>12.1f}")
print(f"{'Min size (chars)':<25} {min(fixed_sizes):>12} {min(sentence_sizes):>12}")
print(f"{'Max size (chars)':<25} {max(fixed_sizes):>12} {max(sentence_sizes):>12}")
print(f"{'Std deviation':<25} {(sum((x - sum(fixed_sizes)/len(fixed_sizes))**2 for x in fixed_sizes) / len(fixed_sizes))**0.5:>12.1f} {(sum((x - sum(sentence_sizes)/len(sentence_sizes))**2 for x in sentence_sizes) / len(sentence_sizes))**0.5:>12.1f}")
print("=" * 50)

### Sentence-Based Chunking: Pros vs Cons

| Pros | Cons |
|---|---|
| Preserves grammatical structure | Chunk sizes are uneven |
| Each chunk is a complete thought | Some sentences may be very short |
| No words are cut in half | Long sentences may exceed model token limits |
| Easy to understand and debug | No overlap between chunks |

**Tip:** In practice, you can combine sentence-based chunking with grouping -- put 2-3 consecutive sentences together to get more context per chunk.

---
## Section 4: Recursive Text Splitting with LangChain

LangChain's `RecursiveCharacterTextSplitter` is the **recommended default** chunking strategy. It works by trying a hierarchy of separators:

```
1. First, try to split on "\n\n" (paragraph boundaries)
2. If chunks are still too large, split on "\n" (line breaks)
3. If still too large, split on ". " (sentences)
4. If still too large, split on " " (words)
5. As a last resort, split on "" (individual characters)
```

This approach **respects the natural structure** of the text -- it tries to keep paragraphs together, then sentences, then words.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a splitter with specific chunk size and overlap
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,       # use character count
    is_separator_regex=False,
)

print("RecursiveCharacterTextSplitter created!")
print(f"  chunk_size: {recursive_splitter._chunk_size}")
print(f"  chunk_overlap: {recursive_splitter._chunk_overlap}")
print(f"  separators: {recursive_splitter._separators}")

In [ ]:
# Split the sample text
recursive_chunks = recursive_splitter.split_text(SAMPLE_TEXT)

print(f"Number of chunks: {len(recursive_chunks)}\n")
for i, chunk in enumerate(recursive_chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()

In [ ]:
# Visualize overlap between consecutive chunks
# We find the shared text between each pair of consecutive chunks

print("Overlap between consecutive chunks:\n")
for i in range(len(recursive_chunks) - 1):
    chunk_a = recursive_chunks[i]
    chunk_b = recursive_chunks[i + 1]
    
    # Find the overlap: the longest suffix of chunk_a that is a prefix of chunk_b
    overlap_text = ""
    for length in range(1, min(len(chunk_a), len(chunk_b)) + 1):
        if chunk_a[-length:] == chunk_b[:length]:
            overlap_text = chunk_a[-length:]
    
    if overlap_text:
        print(f"Chunks {i+1} <-> {i+2}: {len(overlap_text)} chars overlap")
        print(f"  Shared text: \"{overlap_text}\"")
    else:
        print(f"Chunks {i+1} <-> {i+2}: No exact suffix-prefix overlap found")
        # Show the end of chunk_a and start of chunk_b for comparison
        print(f"  End of chunk {i+1}:   \"...{chunk_a[-60:]}\"")
        print(f"  Start of chunk {i+2}: \"{chunk_b[:60]}...\"")
    print()

---
## Section 5: Semantic Chunking

The most advanced approach: instead of splitting by character count or punctuation, we split based on **meaning**.

The idea:
1. Split text into sentences.
2. Embed each sentence.
3. Compute **cosine similarity** between consecutive sentences.
4. When similarity drops significantly, that is a **semantic boundary** -- a place where the topic changes.

This gives us chunks that each cover a coherent topic.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load a lightweight embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded!")

# Step 1: Split into sentences
sentences = re.split(r'(?<=[.!?])\s+', SAMPLE_TEXT)
sentences = [s.strip() for s in sentences if s.strip()]
print(f"Number of sentences: {len(sentences)}")

# Step 2: Embed each sentence
embeddings = model.encode(sentences)
print(f"Embeddings shape: {embeddings.shape}")

# Step 3: Compute cosine similarity between consecutive sentences
similarities = []
for i in range(len(embeddings) - 1):
    sim = cosine_similarity([embeddings[i]], [embeddings[i + 1]])[0][0]
    similarities.append(sim)

print(f"\nSimilarity between consecutive sentences:")
for i, sim in enumerate(similarities):
    marker = "  <-- POTENTIAL BOUNDARY" if sim < 0.45 else ""
    print(f"  Sentence {i+1} <-> {i+2}: {sim:.3f}{marker}")

# Step 4: Identify semantic boundaries (where similarity drops below threshold)
threshold = 0.45
boundaries = [i + 1 for i, sim in enumerate(similarities) if sim < threshold]
print(f"\nSemantic boundaries at sentence indices: {boundaries}")

# Step 5: Create semantic chunks
semantic_chunks = []
start = 0
for boundary in boundaries:
    semantic_chunks.append(" ".join(sentences[start:boundary]))
    start = boundary
semantic_chunks.append(" ".join(sentences[start:]))  # last chunk

print(f"\nNumber of semantic chunks: {len(semantic_chunks)}")
for i, chunk in enumerate(semantic_chunks):
    print(f"\n--- Semantic Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)

In [ ]:
# Visualize the similarity between consecutive sentences

fig, ax = plt.subplots(figsize=(10, 4))

x_labels = [f"{i+1}-{i+2}" for i in range(len(similarities))]
colors = ['red' if sim < threshold else 'steelblue' for sim in similarities]

bars = ax.bar(x_labels, similarities, color=colors, edgecolor='black')
ax.axhline(y=threshold, color='red', linestyle='--', label=f'Threshold ({threshold})')
ax.set_xlabel('Sentence Pair')
ax.set_ylabel('Cosine Similarity')
ax.set_title('Semantic Similarity Between Consecutive Sentences')
ax.legend()
ax.set_ylim(0, 1)

# Add similarity values on top of bars
for bar, sim in zip(bars, similarities):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{sim:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

---
## Section 6: Comparison & Best Practices

Now let's compare all four strategies side by side on the same text.

In [ ]:
import pandas as pd

# Collect all chunking results
all_strategies = {
    "Fixed-Size (200/50)": fixed_chunks,
    "Sentence-Based": sentence_chunks,
    "Recursive (300/50)": recursive_chunks,
    "Semantic": semantic_chunks,
}

# Build comparison table
rows = []
for name, chunks in all_strategies.items():
    sizes = [len(c) for c in chunks]
    rows.append({
        "Strategy": name,
        "Num Chunks": len(chunks),
        "Avg Size": round(sum(sizes) / len(sizes), 1),
        "Min Size": min(sizes),
        "Max Size": max(sizes),
        "Std Dev": round((sum((x - sum(sizes)/len(sizes))**2 for x in sizes) / len(sizes))**0.5, 1),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
# Create a 2x2 comparison figure

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Comparison of Chunking Strategies', fontsize=16, fontweight='bold')

colors_map = {
    "Fixed-Size (200/50)": 'steelblue',
    "Sentence-Based": 'coral',
    "Recursive (300/50)": 'seagreen',
    "Semantic": 'mediumpurple',
}

for ax, (name, chunks) in zip(axes.flatten(), all_strategies.items()):
    sizes = [len(c) for c in chunks]
    color = colors_map[name]
    
    bars = ax.bar(range(1, len(sizes) + 1), sizes, color=color, edgecolor='black', alpha=0.8)
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Chunk Number')
    ax.set_ylabel('Size (chars)')
    ax.set_xticks(range(1, len(sizes) + 1))
    
    avg_size = sum(sizes) / len(sizes)
    ax.axhline(y=avg_size, color='black', linestyle='--', alpha=0.5, label=f'Avg: {avg_size:.0f}')
    ax.legend(fontsize=9)
    
    # Add size labels
    for bar, size in zip(bars, sizes):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
                str(size), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

### When to Use What?

| Strategy | Best For | Speed | Quality |
|---|---|---|---|
| **Fixed-Size** | Simple pipelines, uniform text (e.g., logs, code) | Very fast | Low-Medium |
| **Sentence-Based** | Clean prose, articles, well-punctuated text | Fast | Medium |
| **Recursive (LangChain)** | General-purpose -- the recommended default | Fast | Medium-High |
| **Semantic** | When quality matters most -- Q&A systems, research | Slow | High |

### Decision Guide

```
Do you need high retrieval quality?
  |
  +-- No  --> Fixed-Size chunking (simple and fast)
  |
  +-- Yes --> Is the text well-structured (paragraphs, headings)?
                |
                +-- Yes --> Recursive Text Splitting (best general-purpose)
                |
                +-- No  --> Do you have an embedding model available?
                              |
                              +-- Yes --> Semantic Chunking (highest quality)
                              |
                              +-- No  --> Sentence-Based Chunking
```

### Golden Rules of Chunking

1. **Respect token limits.** For `all-MiniLM-L6-v2`, keep chunks under ~256 tokens (roughly 200-500 characters).
2. **Aim for 200-500 characters** per chunk as a good starting point.
3. **Always use overlap** (10-20% of chunk size) to avoid losing context at boundaries.
4. **Match the strategy to your data.** Structured documents (HTML, Markdown) benefit from recursive splitting. Free-form text benefits from semantic chunking.
5. **Test and iterate.** There is no one-size-fits-all. Try different settings and evaluate retrieval quality on your actual queries.
6. **Think about your queries.** If users ask short, specific questions, use smaller chunks. If they ask broad questions, use larger chunks.

---
## Final Exercise: Your Turn!

**Task:** Take any text you like (a Wikipedia article, a news article, or your own writing), paste it into the cell below, and compare at least 2 chunking strategies.

Questions to answer:
1. How many chunks does each strategy produce?
2. Which strategy gives more meaningful chunks for your text?
3. If you were building a semantic search system for this text, which chunking strategy would you choose and why?

**Bonus:** Try feeding the chunks into the embedding model and performing a similarity search with a query of your choice. Which strategy returns more relevant results?

In [ ]:
# YOUR CODE HERE
# Step 1: Paste your text
my_text = """
Paste your text here...
"""

# Step 2: Apply at least 2 chunking strategies


# Step 3: Compare the results
